In [17]:
import tarfile
import json
import os
import math

import pandas as pd
import numpy as np
import plotly.graph_objects as go

from pathlib import Path
from typing import List, Set
from plotly.subplots import make_subplots


In [ ]:
def extract_uids_and_images_from_tar_files(tar_files_path: str | Path) -> pd.DataFrame:
    """
    Extract all UIDs and images from a set of tar files.
    
    Args:
        tar_files_path (str | Path): Path to directory containing tar files
    
    Returns:
        pd.DataFrame: DataFrame with UIDs and PIL image objects
    """
    import io
    from PIL import Image
    
    tar_files_path = Path(tar_files_path)
    uids_and_images = []
    
    # Get all tar files in the directory
    tar_files = [f for f in tar_files_path.glob("*.tar")]
    
    for tar_path in tar_files:
        try:
            with tarfile.open(tar_path, 'r') as tar:
                # Get all member names
                member_names = tar.getnames()
                
                # Filter to get just the JSON files
                json_files = [name for name in member_names if name.endswith('.json')]
                
                for json_file in json_files:
                    # Extract and read the JSON file to get the UID
                    json_member = tar.getmember(json_file)
                    json_fileobj = tar.extractfile(json_member)
                    
                    if json_fileobj is not None:
                        try:
                            metadata = json.load(json_fileobj)
                            uid = metadata.get("uid")
                            
                            if uid:
                                # Find the corresponding image file
                                image_name = json_file.replace('.json', '.jpg')
                                if image_name in member_names:
                                    image_member = tar.getmember(image_name)
                                    image_fileobj = tar.extractfile(image_member)
                                    
                                    if image_fileobj is not None:
                                        # Load the image into a PIL Image object
                                        image_data = image_fileobj.read()
                                        image = Image.open(io.BytesIO(image_data))
                                        
                                        # Add UID and image to the list
                                        uids_and_images.append({
                                            'uid': uid,
                                            'image': image
                                        })
                        except json.JSONDecodeError:
                            print(f"Error decoding JSON file {json_file} in {tar_path}")
        except tarfile.TarError as e:
            print(f"Error reading {tar_path}: {e}")
    
    # Create DataFrame from the list of dictionaries
    return pd.DataFrame(uids_and_images)


def filter_dataframe_by_uids(df: pd.DataFrame, filter_uids: List[str]) -> pd.DataFrame:
    """
    Filter DataFrame based on a given list of UIDs.
    
    Args:
        df (pd.DataFrame): DataFrame with UIDs and images
        filter_uids (List[str]): UIDs to filter by
    
    Returns:
        pd.DataFrame: Filtered DataFrame
    """
    filter_set = set(filter_uids)
    return df[df['uid'].isin(filter_set)]

# Example path to tar files
tar_directory = "/home/fbernardi/Documents/fair_spoke_8/data_quality_pipeline/test/data"

# Extract all UIDs and images
df_images = extract_uids_and_images_from_tar_files(tar_directory)
print(f"Total images found: {len(df_images)}")


In [3]:
semdedup_metadata_path = '/home/fbernardi/Documents/fair_spoke_8/data_quality_pipeline/src/made/semdedup/data'

# Distances to centroids
dist_to_cent = np.load(
    os.path.join(
        semdedup_metadata_path,
        'clustering', 
        'dist_to_cent.npy'
    )
)

# Vectors of centroids
centroids = np.load(
    os.path.join(
        semdedup_metadata_path,
        'clustering', 
        'kmeans_centroids.npy'
    )
)

nearest_cent = np.load(
    os.path.join(
        semdedup_metadata_path,
        'clustering', 
        'nearest_cent.npy'
    )
)

In [14]:
cluster_id = 50
eps = 0.1

pruning_table = pd.read_pickle(
    os.path.join(
        semdedup_metadata_path,
        'dataframes',
        f'cluster_{cluster_id}.pkl'
    )
)

cluster = np.load(
    os.path.join(
        semdedup_metadata_path,
        'sorted_clusters', 
        f'cluster_{cluster_id}.npy'
    )
)

In [ ]:
true_uids  = pruning_table[pruning_table[f"eps={eps}"]==True]["cluster_uids"].to_list()
false_uids  = pruning_table[pruning_table[f"eps={eps}"]==False]["cluster_uids"].to_list()

print(
    f"\nTrue UIDs: {len(true_uids)}",
    f"\nFalse UIDs: {len(false_uids)}",
    f"\nTotal UIDs: {len(true_uids) + len(false_uids)}",
)

In [6]:
def plot_images_matrix_plotly_chunked(df_filtered, chunk_size=50, max_cols=6, 
                                     width_per_image=200, height_per_image=200, 
                                     show_progress=True):
    """
    Plot PIL images with UIDs as titles using Plotly, handling large datasets by chunking.
    
    Args:
        df_filtered: DataFrame with 'image' and 'uid' columns
        chunk_size: Number of images per plot/chunk
        max_cols: Maximum number of columns in each grid
        width_per_image: Width of each image in pixels
        height_per_image: Height of each image in pixels
        show_progress: Whether to print progress information
    """
    if (chunk_size is None) or (chunk_size < 50):
        chunk_size = 50
    
    total_images = len(df_filtered)
    if total_images == 0:
        print("No images to display")
        return
        
    # Calculate number of chunks needed
    num_chunks = math.ceil(total_images / chunk_size)
    
    if show_progress:
        print(f"Total images: {total_images}")
        print(f"Chunk size: {chunk_size}")
        print(f"Number of plots to generate: {num_chunks}")
        print("-" * 50)
    
    # Process each chunk
    for chunk_idx in range(num_chunks):
        start_idx = chunk_idx * chunk_size
        end_idx = min(start_idx + chunk_size, total_images)
        
        # Get current chunk of data
        chunk_df = df_filtered.iloc[start_idx:end_idx].copy()
        chunk_size_actual = len(chunk_df)
        
        if show_progress:
            print(f"Processing chunk {chunk_idx + 1}/{num_chunks}: "
                  f"images {start_idx + 1}-{end_idx} ({chunk_size_actual} images)")
        
        # Calculate grid dimensions for this chunk
        n_cols = min(max_cols, chunk_size_actual)
        n_rows = math.ceil(chunk_size_actual / n_cols)
        
        # Create subplot titles with UIDs
        subplot_titles = []
        for _, row in chunk_df.iterrows():
            uid_short = row['uid'][:8] + '...' if len(row['uid']) > 8 else row['uid']
            subplot_titles.append(uid_short)
        
        # Fill remaining titles with empty strings
        subplot_titles.extend([''] * (n_rows * n_cols - chunk_size_actual))
        
        # Create subplots for this chunk
        fig = make_subplots(
            rows=n_rows, 
            cols=n_cols,
            subplot_titles=subplot_titles,
            vertical_spacing=0.08,
            horizontal_spacing=0.05
        )
        
        # Add images to subplots
        for i, (_, row) in enumerate(chunk_df.iterrows()):
            plot_row = (i // n_cols) + 1  # Plotly uses 1-indexed
            plot_col = (i % n_cols) + 1
            
            # Convert PIL image to numpy array
            img_array = np.array(row['image'])
            
            # Add image to subplot
            fig.add_trace(
                go.Image(z=img_array),
                row=plot_row, col=plot_col
            )
        
        # Update layout
        chunk_title = f"Image Matrix - Chunk {chunk_idx + 1}/{num_chunks} (Images {start_idx + 1}-{end_idx})"
        fig.update_layout(
            title=chunk_title,
            showlegend=False,
            width=n_cols * width_per_image,
            height=n_rows * height_per_image + 150  # Extra space for titles
        )
        
        # Remove axes for all subplots
        for i in range(1, n_rows + 1):
            for j in range(1, n_cols + 1):
                fig.update_xaxes(showticklabels=False, showgrid=False, zeroline=False, row=i, col=j)
                fig.update_yaxes(showticklabels=False, showgrid=False, zeroline=False, row=i, col=j)
        
        # Show the plot for this chunk
        fig.show()
        
        if show_progress and chunk_idx < num_chunks - 1:
            print(f"Chunk {chunk_idx + 1} completed. Preparing next chunk...")
            print()

def plot_images_from_filter(df_images, uid_filter, chunk_size=50, max_cols=6, 
                           width_per_image=200, height_per_image=200):
    """
    Convenience function to filter and plot images in chunks.
    
    Args:
        df_images: DataFrame with 'image' and 'uid' columns
        uid_filter: List/Series of UIDs to filter by
        chunk_size: Number of images per plot/chunk
        max_cols: Maximum number of columns in each grid
        width_per_image: Width of each image in pixels
        height_per_image: Height of each image in pixels
    """
    # Filter the dataframe
    filtered_df = df_images[df_images['uid'].isin(uid_filter)]
    
    print(f"Filtered {len(filtered_df)} images from {len(df_images)} total images")
    print(f"Filter contains {len(uid_filter)} unique UIDs")
    print()
    
    # Plot in chunks
    plot_images_matrix_plotly_chunked(
        filtered_df, 
        chunk_size=chunk_size, 
        max_cols=max_cols,
        width_per_image=width_per_image, 
        height_per_image=height_per_image
    )

In [7]:
def plot_comparison_chunks(df_images, true_uids, false_uids, chunk_size=50, max_cols=6):
    """
    Plot both true and false images in chunks for comparison.
    
    Args:
        df_images: DataFrame with image data
        true_uids: UIDs for true/positive images
        false_uids: UIDs for false/negative images
        chunk_size: Number of images per chunk
        max_cols: Maximum columns per grid
    """
    print("=== PLOTTING TRUE IMAGES ===")
    plot_images_from_filter(df_images, true_uids, chunk_size, max_cols)
    
    print("\n" + "="*50)
    print("=== PLOTTING FALSE IMAGES ===")
    plot_images_from_filter(df_images, false_uids, chunk_size, max_cols)

def get_chunk_info(total_images, chunk_size, max_cols=6):
    """
    Calculate and display information about how images will be chunked.
    
    Args:
        total_images: Total number of images
        chunk_size: Images per chunk
        max_cols: Maximum columns per grid
    """
    num_chunks = math.ceil(total_images / chunk_size)
    
    print(f"Chunking Information:")
    print(f"- Total images: {total_images}")
    print(f"- Chunk size: {chunk_size}")
    print(f"- Number of chunks: {num_chunks}")
    print(f"- Max columns per grid: {max_cols}")
    
    for i in range(num_chunks):
        start_idx = i * chunk_size
        end_idx = min(start_idx + chunk_size, total_images)
        images_in_chunk = end_idx - start_idx
        rows_needed = math.ceil(images_in_chunk / max_cols)
        
        print(f"  Chunk {i+1}: {images_in_chunk} images, {rows_needed} rows x {min(max_cols, images_in_chunk)} cols")

In [ ]:
#  Rejected samples
plot_images_from_filter(df_images, true_uids, chunk_size=10, max_cols=6)

In [ ]:
# Kept samples
plot_images_from_filter(df_images, false_uids, chunk_size=60, max_cols=8)